### Name: Blessing Adeniji
### Degree: MSc Artifical Intelligence Online
### Capstone Project: AI-Generated Text Detection - Deepfakes
Purpose: Fine-tune Ettin encoder-400m and decoder-400m on MAGE, evaluate on all four test sets. Tests whether the encoder/decoder holds at 6x size.

In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
print(torch.version.cuda)

CUDA available: True
12.8


In [1]:
import os
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score 

# Metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1_score(labels, preds)}


In [2]:
# Encoder-400m: fine-tune on MAGE
mage_train_dataset = pd.read_csv("data_splits/MAGE_train.csv")
mage_validation_dataset = pd.read_csv("data_splits/MAGE_val.csv")

tokenizer = AutoTokenizer.from_pretrained("jhu-clsp/ettin-encoder-400m")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)

mage_train_ds = Dataset.from_pandas(mage_train_dataset).map(tokenize, batched=True)
mage_val_ds = Dataset.from_pandas(mage_validation_dataset).map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained("jhu-clsp/ettin-encoder-400m", num_labels=2)

training_args = TrainingArguments(
    output_dir="models/ettin_encoder400m_mage",
    num_train_epochs=3,
    per_device_train_batch_size=4, # 400M params on 16gb - smaller batch
    gradient_accumulation_steps=4,       # 4x4 = effective batch 16, matching the 68m MAGE runs
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    bf16=True,
    report_to="none",
)

trainer = Trainer(
    model=model, 
    args=training_args, 
    train_dataset=mage_train_ds, 
    eval_dataset=mage_val_ds,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer), 
    compute_metrics=compute_metrics
)

trainer.train()

Map:   0%|          | 0/130645 [00:00<?, ? examples/s]

Map:   0%|          | 0/27995 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: jhu-clsp/ettin-encoder-400m
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.576683,0.115085,0.967244,0.967115
2,0.303836,0.143737,0.971102,0.970946
3,0.000044,0.199214,0.972959,0.972806


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=24498, training_loss=0.31545411401042567, metrics={'train_runtime': 18197.5397, 'train_samples_per_second': 21.538, 'train_steps_per_second': 1.346, 'total_flos': 3.1335334000459475e+17, 'train_loss': 0.31545411401042567, 'epoch': 3.0})

In [3]:
# Build 4x4 matrix
import os

def save_results_to_csv(model_name, trained_on, tested_on, results):
    # Create a one-row table with the results
    row = pd.DataFrame({
        'model_name': [model_name],
        'trained_on': [trained_on],
        'tested_on': [tested_on],
        'accuracy': [results['eval_accuracy']],
        'f1': [results['eval_f1']],
        'loss': [results['eval_loss']]
    })

    # Append to results and create if it doesn't exist
    file_exists = os.path.isfile('all_models_evaluation_results.csv')
    row.to_csv('all_models_evaluation_results.csv', mode='a', header=not file_exists, index=False)
    print("Saved:", model_name, trained_on, tested_on)

In [4]:
# Save the ettin-encoder-400m model and tokenizer
model.save_pretrained("models/ettin_encoder400m_mage_final")
tokenizer.save_pretrained("models/ettin_encoder400m_mage_final")

# Load and tokenize all 4 test sets with the encoder-400m tokenizer
mage_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/MAGE_test.csv")).map(tokenize, batched=True)
abstracts_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/ChatGPT-Research-Abstracts_test.csv")).map(tokenize, batched=True)
wiki_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/GPT-Wiki-intro_test.csv")).map(tokenize, batched=True)
raid_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/RAID_test.csv")).map(tokenize, batched=True)

# Evaluate - labels match datasets line by line
save_results_to_csv("ettin-encoder-400m", "MAGE", "MAGE", trainer.evaluate(mage_test_ds))
save_results_to_csv("ettin-encoder-400m", "MAGE", "Abstracts", trainer.evaluate(abstracts_test_ds))
save_results_to_csv("ettin-encoder-400m", "MAGE", "Wiki", trainer.evaluate(wiki_test_ds))
save_results_to_csv("ettin-encoder-400m", "MAGE", "RAID", trainer.evaluate(raid_test_ds))

print("\nEncoder-400m evaluation complete")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/27996 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Map:   0%|          | 0/44021 [00:00<?, ? examples/s]

...

Saved: ettin-encoder-400m MAGE Abstracts


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000044,0.693849,3,0.824222,0.838446


Saved: ettin-encoder-400m MAGE Wiki


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000044,1.567865,3,0.777833,0.789587


Saved: ettin-encoder-400m MAGE RAID

Encoder-400m evaluation complete


In [5]:
# Decoder-400m: fine-tune on MAGE
mage_train_dataset = pd.read_csv("data_splits/MAGE_train.csv")
mage_validation_dataset = pd.read_csv("data_splits/MAGE_val.csv")

tokenizer = AutoTokenizer.from_pretrained("jhu-clsp/ettin-decoder-400m")

# Decoders have no padding token by default, so use the end-of-sequence token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)

mage_train_ds = Dataset.from_pandas(mage_train_dataset).map(tokenize, batched=True)
mage_val_ds = Dataset.from_pandas(mage_validation_dataset).map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained("jhu-clsp/ettin-decoder-400m", num_labels=2)

# Decoder - this is the model's config for the token that is the pad token
model.config.pad_token_id = tokenizer.pad_token_id

training_args = TrainingArguments(
    output_dir="models/ettin_decoder400m_mage",
    num_train_epochs=3,
    per_device_train_batch_size=4, # 400M params on 16gb - smaller batch
    gradient_accumulation_steps=4, # 4x4 = effective batch 16, matching the 68m MAGE runs
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    bf16=True,
    report_to="none",
)

trainer = Trainer(
    model=model, 
    args=training_args, 
    train_dataset=mage_train_ds, 
    eval_dataset=mage_val_ds,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer), 
    compute_metrics=compute_metrics
)

trainer.train()

config.json:   0%|          | 0.00/2.12k [00:00<?, ?B/s]

c:\Users\Blessing\capstone2026-adeniji-detecting-Ai-generated-text\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Blessing\.cache\huggingface\hub\models--jhu-clsp--ettin-decoder-400m. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/22.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=24498, training_loss=0.3594659920740793, metrics={'train_runtime': 18284.6658, 'train_samples_per_second': 21.435, 'train_steps_per_second': 1.34, 'total_flos': 3.1335333818413056e+17, 'train_loss': 0.3594659920740793, 'epoch': 3.0})

In [ ]:
# Save the decoder-400m model and tokenizer
model.save_pretrained("models/ettin_decoder400m_mage_final")
tokenizer.save_pretrained("models/ettin_decoder400m_mage_final")

# Load and tokenize all 4 test sets with the decoder-400m tokenizer
mage_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/MAGE_test.csv")).map(tokenize, batched=True)
abstracts_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/ChatGPT-Research-Abstracts_test.csv")).map(tokenize, batched=True)
wiki_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/GPT-Wiki-intro_test.csv")).map(tokenize, batched=True)
raid_test_ds = Dataset.from_pandas(pd.read_csv("data_splits/RAID_test.csv")).map(tokenize, batched=True)

# Evaluate - labels match datasets line by line
save_results_to_csv("ettin-decoder-400m", "MAGE", "MAGE", trainer.evaluate(mage_test_ds))
save_results_to_csv("ettin-decoder-400m", "MAGE", "Abstracts", trainer.evaluate(abstracts_test_ds))
save_results_to_csv("ettin-decoder-400m", "MAGE", "Wiki", trainer.evaluate(wiki_test_ds))
save_results_to_csv("ettin-decoder-400m", "MAGE", "RAID", trainer.evaluate(raid_test_ds))

print("\nDecoder-400m evaluation complete")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/27996 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/45000 [00:00<?, ? examples/s]

Map:   0%|          | 0/44021 [00:00<?, ? examples/s]

...

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000083,0.409647,3,0.885000,0.884803


Saved: ettin-decoder-400m MAGE Wiki


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000083,1.156641,3,0.773903,0.792520


Saved: ettin-decoder-400m MAGE RAID

Decoder-400m evaluation complete


: 